# Customer Segmentation using K-Means

Run this notebook in Google Colab. It loads the sample CSV from this GitHub repository, prepares the data, applies K-Means, evaluates the clusters, and visualizes them.

**Dataset note:** The 500 customer records are synthetic sample data generated for demonstration. They are not real company/customer records.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from IPython.display import display

# Load sample data from the GitHub repository
DATA_URL = 'https://raw.githubusercontent.com/kushank-garg/customer-segmentation/main/data/customer_data.csv'
df = pd.read_csv(DATA_URL)
print('Dataset shape:', df.shape)
display(df.head())
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
features = ['Age', 'AnnualIncome_k', 'SpendingScore', 'AnnualVisits']
X = df[features].copy().fillna(df[features].median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Features standardized:', features)

In [ ]:
# Elbow method: compare inertia for K values from 2 through 8
k_values = range(2, 9)
inertias = []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertias, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia')
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# Fit K-Means with K=4 for this four-pattern synthetic demonstration dataset
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)
silhouette = silhouette_score(X_scaled, df['Cluster'])
print(f'Silhouette score (K=4): {silhouette:.3f}')

# Summarize average feature values and customer counts by cluster
profile = df.groupby('Cluster')[features].mean().round(2)
profile['CustomerCount'] = df['Cluster'].value_counts().sort_index()
display(profile)

In [ ]:
# Assign descriptive labels based on each cluster's actual income/spending averages.
# Cluster IDs are arbitrary, so do not assume a particular ID always means a particular segment.
income_mid = profile['AnnualIncome_k'].median()
spending_mid = profile['SpendingScore'].median()

def describe_segment(row):
    income_level = 'High Income' if row['AnnualIncome_k'] >= income_mid else 'Low Income'
    spending_level = 'High Spending' if row['SpendingScore'] >= spending_mid else 'Low Spending'
    return f'{income_level} - {spending_level}'

cluster_names = {int(cluster): describe_segment(row) for cluster, row in profile.iterrows()}
df['Segment'] = df['Cluster'].map(cluster_names)
profile['SegmentName'] = [cluster_names[int(cluster)] for cluster in profile.index]
print('Cluster interpretation:')
display(profile[['SegmentName', 'CustomerCount', 'AnnualIncome_k', 'SpendingScore', 'Age', 'AnnualVisits']])
print('\nExample customer assignments:')
display(df[['CustomerID', 'Cluster', 'Segment']].head(10))

In [ ]:
# PCA is used only to visualize the four-dimensional data in two dimensions.
points = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plt.figure(figsize=(10, 6))
for cluster in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster
    plt.scatter(points[mask, 0], points[mask, 1],
                label=f"Cluster {cluster}: {cluster_names[int(cluster)]}", alpha=0.7)
plt.title('Customer Segmentation using K-Means (PCA View)')
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('Principal Component 2 (PC2)')
plt.legend(title='Customer segments', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Save downloadable result files in the Colab session
df.to_csv('segmented_customers.csv', index=False)
profile.to_csv('segment_profiles.csv')
print('Created segmented_customers.csv and segment_profiles.csv')
from google.colab import files
files.download('segment_profiles.csv')

## Notes and limitations
- The dataset is synthetic and for demonstration only.
- K=4 is used because the sample data was generated with four broad patterns; this is not a claim that K=4 is universally optimal.
- K-Means is sensitive to feature scaling, outliers, and the chosen number of clusters.
- PCA is for visualization; it does not perform the clustering.